## Import Libraries

In [1]:
import glob, re
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from tropycal import tracks

/Users/xochitlhidalgo/miniforge3/envs/coding_env/lib/python3.13/site-packages/tropycal/_version.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution


## Create functions to find date to match with track location and calculate the haversine distance.

In [ ]:
def extract_time(fname):
    date = re.findall(r'\d{8}', fname)[0]
    return pd.to_datetime(date, format='%Y%m%d')

## Haversine Distance
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

## Create function to select data within 500km of the storm center and sum rain and hit counts.

In [ ]:
def storm_data(name, year, prism_data):
    radius_km = 500
    deg_buffer = 5

    ## load files
    prism_data = sorted(prism_data)

    ds_list = []
    times = []

    for f in prism_data:
        ds_i = xr.open_dataset(f)
        ds_list.append(ds_i)
        times.append(extract_time(f) + pd.Timedelta(hours=12))

    ds = xr.concat(ds_list, dim='time')
    ds = ds.assign_coords(time=('time', times))

    precip = ds['Band1']
    lat = ds['lat'].values
    lon = ds['lon'].values


    ## load track from hurdat2
    basin = tracks.TrackDataset(basin='north_atlantic', source='hurdat')
    storm = basin.get_storm((name, year))
    df = storm.to_dataframe()
    df['time'] = pd.to_datetime(df['time'])
    df = df[df['time'].dt.hour == 12]

    ## initialize arrays
    rain_sum = np.zeros((len(lat), len(lon)))
    hit_count = np.zeros((len(lat), len(lon)))


    ## loop through and select data within 500km of storm center
    for _, row in df.iterrows():

        clat, clon = row['lat'], row['lon']

        ## skip storm positions outside PRISM domain
        if not (lat.min() <= clat <= lat.max() and lon.min() <= clon <= lon.max()):
            continue

        t = row['time']

        if t not in ds.time.values:
            continue

        rain_day = precip.sel(time=t).fillna(0)

        ## spatial subset (for speed)
        lat_mask = (lat >= clat - deg_buffer) & (lat <= clat + deg_buffer)
        lon_mask = (lon >= clon - deg_buffer) & (lon <= clon + deg_buffer)

        if lat_mask.sum() == 0 or lon_mask.sum() == 0:
            continue

        sub_rain = rain_day.values[np.ix_(lat_mask, lon_mask)]
        sub_rain = np.nan_to_num(sub_rain, nan=0.0)

        sub_lat = lat[lat_mask]
        sub_lon = lon[lon_mask]
        lon2d, lat2d = np.meshgrid(sub_lon, sub_lat)

        dist = haversine_km(clat, clon, lat2d, lon2d)
        mask = dist <= radius_km

        ## extract subset views
        rs = rain_sum[np.ix_(lat_mask, lon_mask)]
        hc = hit_count[np.ix_(lat_mask, lon_mask)]

        ## update data
        rs[mask] += sub_rain[mask]
        hc[mask] += 1

        ## add to arrays
        rain_sum[np.ix_(lat_mask, lon_mask)] = rs
        hit_count[np.ix_(lat_mask, lon_mask)] = hc
    

    return rain_sum, hit_count, lat, lon

## Read in csv with hurricane information

In [50]:
hur_dat = pd.read_csv('Hurricanes.csv', skiprows=1)
hur_dat = hur_dat.rename(columns={'Unnamed: 0':'Name','Unnamed: 1':'start','Unnamed: 2':'end', 'Unnamed: 3':'Category'}).dropna(how='all').dropna(axis=1, how='all')
hur_dat['start'] = pd.to_datetime(hur_dat['start'])
hur_dat['end'] = pd.to_datetime(hur_dat['end'])
hur_dat['year'] = hur_dat['start'].dt.year
hur_dat

/var/folders/z6/1637m0dd1p1dn2n4wrbvj6kw0000gn/T/ipykernel_90995/3877447472.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  hur_dat['start'] = pd.to_datetime(hur_dat['start'])
/var/folders/z6/1637m0dd1p1dn2n4wrbvj6kw0000gn/T/ipykernel_90995/3877447472.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  hur_dat['end'] = pd.to_datetime(hur_dat['end'])


,Name,start,end,Category,year
0,Milton,2024-10-09,2024-10-10,3.0,2024
1,Helene,2024-09-26,2024-09-28,4.0,2024
2,Francine,2024-09-10,2024-09-12,2.0,2024
3,Debby,2024-08-04,2024-08-08,1.0,2024
4,Beryl,2024-07-08,2024-07-09,1.0,2024
5,Idalia,2023-08-30,2023-08-31,3.0,2023
6,Nicole,2022-11-09,2022-11-11,1.0,2022
7,Ian,2022-09-27,2022-09-30,4.0,2022
8,Nicholas,2021-09-13,2021-09-16,1.0,2021
9,Ida,2021-08-28,2021-09-01,4.0,2021


## For each storm in list, call function and save pertinent information (rain, hits, lat/lon, storm names, year, and category)

In [51]:
storm_list = hur_dat.to_dict('records')

rain = []
hits = []
lats = []
lons = []
storm_names = []
years = []
categories = []

all_files = sorted(glob.glob('rain_data/*.nc'))

for storm in storm_list:

    name = storm['Name']
    year = storm['year']
    start = storm['start']
    end = storm['end']
    category = storm['Category']

    # Select files by DATE (robust)
    files = []
    for f in all_files:
        try:
            t = extract_time(f)
        except:
            continue

        if start <= t <= end:
            files.append(f)

    # Debug check
    if len(files) == 0:
        print(f'No files for {name} {year}')
        continue

    try:
        rain_sum, hit_count, lat, lon = storm_data(name, year, files)
    except Exception as e:
        print(f'Skipping {name} {year}: {e}')
        continue

    if np.nansum(rain_sum) == 0:
        print(f'No rainfall for {name} {year}')
        continue
    

    rain.append(rain_sum)
    hits.append(hit_count)
    lats.append(lat)
    lons.append(lon)
    storm_names.append(name)
    years.append(year)
    categories.append(category)

--> Starting to read in HURDAT2 data
--> Completed reading in HURDAT2 data (6.77 seconds)
--> Starting to read in HURDAT2 data
--> Completed reading in HURDAT2 data (3.51 seconds)
--> Starting to read in HURDAT2 data
--> Completed reading in HURDAT2 data (5.38 seconds)
--> Starting to read in HURDAT2 data
--> Completed reading in HURDAT2 data (4.41 seconds)
--> Starting to read in HURDAT2 data
--> Completed reading in HURDAT2 data (4.48 seconds)
--> Starting to read in HURDAT2 data
--> Completed reading in HURDAT2 data (3.91 seconds)
--> Starting to read in HURDAT2 data
--> Completed reading in HURDAT2 data (3.71 seconds)
--> Starting to read in HURDAT2 data
--> Completed reading in HURDAT2 data (3.57 seconds)
--> Starting to read in HURDAT2 data
--> Completed reading in HURDAT2 data (3.39 seconds)
--> Starting to read in HURDAT2 data
--> Completed reading in HURDAT2 data (3.91 seconds)
--> Starting to read in HURDAT2 data
--> Completed reading in HURDAT2 data (3.23 seconds)
--> Starti

## Convert data to nc file for easier use

In [53]:

rain_arr = np.stack(rain)   # shape: (storm, lat, lon)
hits_arr = np.stack(hits)   # shape: (storm, lat, lon)

storm_arr = np.array(storm_names)
year_arr = np.array(years)

lat = lats[0]
lon = lons[0]

ds = xr.Dataset(
    {
        'rain': (('storm', 'lat', 'lon'), rain_arr),
        'hits': (('storm', 'lat', 'lon'), hits_arr),
    },
    coords={
        'storm': storm_arr,          # shape (storm,)
        'year': ('storm', year_arr), # shape (storm,)
        'cat': ('storm', categories),# shape (storm,)
        'lat': lat,                  # shape (lat,)
        'lon': lon                   # shape (lon,)
    }
)


ds.to_netcdf('storm_rainfall.nc')


In [55]:
ds = xr.open_dataset('storm_rainfall.nc')
ds.cat

<xarray.DataArray 'cat' (storm: 53)> Size: 424B
[53 values with dtype=float64]
Coordinates:
  * storm    (storm) <U9 2kB 'Milton' 'Helene' 'Francine' ... 'Fran' 'Bertha'
    year     (storm) int64 424B ...
    cat      (storm) float64 424B ...